In [1]:
%%writefile anti_entropy.py
from mpi4py import MPI
import time
import random

# ==========================================
# 1. FUNCIONES PARA DIFUSIÓN DE VALOR (ANTI-ENTROPÍA)
# ==========================================
# El estado guarda el valor y su timestamp: {'value': string, 'timestamp': float}

def selectNeighbor(my_rank, total_nodes):
    # q <- selectNeighbor()
    target = my_rank
    while target == my_rank:
        target = random.randint(0, total_nodes - 1)
    return target

def prepareRequest(state, q):
    # msg <- prepareRequest(state, q)
    # Solicitamos la sincronización enviando nuestro estado actual
    return state.copy()

def prepareReply(state, req, q):
    # rep <- prepareReply(state, req, q)
    # Respondemos al mensaje 'req' de 'q' enviando nuestro estado actual
    return state.copy()

def mergeRequest(state, req, q):
    # state <- mergeRequest(state, req, q)
    # Procesamos el request de 'q': si su valor es más nuevo, nos actualizamos
    if req['timestamp'] > state['timestamp']:
        return req.copy()
    return state.copy()

def mergeReply(state, rep, q):
    # state <- mergeReply(state, rep, q)
    # Procesamos la respuesta de 'q': si su valor es más nuevo, nos actualizamos
    if rep['timestamp'] > state['timestamp']:
        return rep.copy()
    return state.copy()


# ==========================================
# 2. ESQUEMA GENERAL (ALGORITMO 3)
# ==========================================
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

TAG_REQUEST = 1
TAG_REPLY = 2
delta = 0.1
SIMULATION_TIME = 2.0

# -- on initialization --
# state is initialized
if rank == 0:
    state = {'value': "¡Valor Actualizado v2.0!" , 'timestamp': time.time()}
else:
    state = {'value': "valor_viejo", 'timestamp': 0.0}

start_simulation = time.time()
# set timeout \Delta
last_timeout = start_simulation - delta 

# Ciclo asíncrono
while time.time() - start_simulation < SIMULATION_TIME:
    
    # -- on timeout --
    if (time.time() - last_timeout) >= delta:
        q = selectNeighbor(rank, size)
        msg = prepareRequest(state, q)
        
        # send <REQUEST, msg, p> to q
        comm.send(msg, dest=q, tag=TAG_REQUEST)
        # set timeout \Delta
        last_timeout = time.time()

    # -- Recepción de mensajes --
    status = MPI.Status()
    if comm.iprobe(source=MPI.ANY_SOURCE, tag=MPI.ANY_TAG, status=status):
        # 'sender' representa a la variable 'q' (el nodo vecino) en el pseudocódigo
        sender = status.Get_source()
        tag = status.Get_tag()
        
        if tag == TAG_REQUEST:
            # -- on receive <REQUEST, req, q> --
            req = comm.recv(source=sender, tag=TAG_REQUEST)
            rep = prepareReply(state, req, sender)
            
            # send <REPLY, rep, p> to q
            comm.send(rep, dest=sender, tag=TAG_REPLY)
            
            state = mergeRequest(state, req, sender)
            
        elif tag == TAG_REPLY:
            # -- on receive <REPLY, rep, q> --
            rep = comm.recv(source=sender, tag=TAG_REPLY)
            state = mergeReply(state, rep, sender)

# ==========================================
# 3. RECOLECCIÓN DE RESULTADOS
# ==========================================
final_states = comm.gather(state, root=0)

if rank == 0:
    print("\n--- RESULTADOS DE DIFUSIÓN DE VALOR ---")
    for i, final_state in enumerate(final_states):
        print(f"Nodo {i} tiene el valor: '{final_state['value']}'")

Overwriting anti_entropy.py


In [2]:
%%writefile rumor_mongering.py
import time
import random
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()


# 1. Definición de Estados
SUSCEPTIBLE = "SUSCEPTIBLE"
INFECTED = "INFECTED"
REMOVED = "REMOVED"

# Tags para diferenciar los tipos de mensajes
TAG_PUSH = 1
TAG_REPLY = 2

# Variables iniciales (Estado base para todos)
state = SUSCEPTIBLE
value = None
k = 5          # Contador.
counter = 0
delta = 0.1    # Timeout (0.1 segundos) Tamano de la ronda.

# --- update(v) ---
# El nodo 0 será el origen del chisme
if rank == 0:
    value = "¡El rumor ha comenzado!"
    state = INFECTED
    counter = k

last_timeout = time.time()
start_simulation = time.time()

# BUCLE DE EVENTOS
# Como es una simulación distribuida, la dejamos correr por 3 segundos 
# para dar tiempo a que el rumor se propague a los 100+ nodos.
while time.time() - start_simulation < 3.0:

    # --- on timeout ---
    if state == INFECTED and (time.time() - last_timeout) >= delta:
        # q <- random(P) (que no sea yo mismo)
        q = random.choice([i for i in range(size) if i != rank])
        
        # send <PUSH, p, value> to q
        # Usamos comm.send para enviar un diccionario
        comm.send({'p': rank, 'v': value}, dest=q, tag=TAG_PUSH)
        
        # set timeout delta (reiniciar temporizador)
        last_timeout = time.time()

    # --- on receive <PUSH, q, v> ---
    # iprobe devuelve True si hay un mensaje de tipo PUSH esperando
    if comm.iprobe(source=MPI.ANY_SOURCE, tag=TAG_PUSH):
        msg = comm.recv(source=MPI.ANY_SOURCE, tag=TAG_PUSH)
        q_sender = msg['p']
        v_received = msg['v']
        
        # send <REPLY, state> to q (al que me mandó el push)
        comm.send({'s': state}, dest=q_sender, tag=TAG_REPLY)
        
        if state == SUSCEPTIBLE:
            value = v_received
            state = INFECTED
            counter = k
            last_timeout = time.time() # Inicia su propio timeout

    # --- on receive <REPLY, s> ---
    if comm.iprobe(source=MPI.ANY_SOURCE, tag=TAG_REPLY):
        msg = comm.recv(source=MPI.ANY_SOURCE, tag=TAG_REPLY)
        s_reply = msg['s']
        
        # Si me responden y yo sigo infectado...
        if s_reply != SUSCEPTIBLE and state == INFECTED:
            counter -= 1
            if counter == 0:
                state = REMOVED

# --- FIN DE LA SIMULACIÓN: RECOPILACIÓN DE DATOS ---
# Usamos gather para que el proceso 0 junte los estados finales y arme el reporte
estados_finales = comm.gather(state, root=0)

if rank == 0:
    total_susceptibles = estados_finales.count(SUSCEPTIBLE)
    total_infectados = estados_finales.count(INFECTED)
    total_removidos = estados_finales.count(REMOVED)
    
    print("=== REPORTE FINAL DE RUMOR MONGERING ===")
    print(f"Total de nodos (procesos) simulados: {size}")
    print(f"Nodos que nunca escucharon el rumor (SUSCEPTIBLES): {total_susceptibles}")
    print(f"Nodos que siguen esparciendo (INFECTADOS): {total_infectados}")
    print(f"Nodos que perdieron el interés (REMOVIDOS): {total_removidos}")
    print("========================================")

Overwriting rumor_mongering.py


In [ ]:
!mpiexec --oversubscribe -n 150 python3 anti_entropy.py

In [ ]:
!mpiexec --oversubscribe -n 100 python3 rumor_mongering.py